# Ch. 4 — Linear Neural Networks for Classification
*Dive into Deep Learning (PyTorch) — code + minimal notes*

> **Note:** the book uses Fashion-MNIST (`4.2`). This environment has no network access to download it, so a small synthetic multi-class dataset stands in below — same shapes, same training code. Swap in `torchvision.datasets.FashionMNIST` locally if you want the real thing; nothing else changes.

## 4.1 Softmax Regression — setup
- classification: pick 1-of-K classes, not a continuous number
- softmax turns raw scores ("logits") into a probability distribution: non-negative, sums to 1

In [1]:
import torch
from torch import nn
from torch.utils import data

torch.manual_seed(0)

num_inputs, num_outputs, num_examples = 20, 5, 1000

# synthetic stand-in for Fashion-MNIST: each class = points near its own random center
centers = torch.randn(num_outputs, num_inputs) * 3
labels = torch.randint(0, num_outputs, (num_examples,))
features = centers[labels] + torch.randn(num_examples, num_inputs)

dataset = data.TensorDataset(features, labels)
train_iter = data.DataLoader(dataset, batch_size=32, shuffle=True)
features.shape, labels.shape

(torch.Size([1000, 20]), torch.Size([1000]))

## 4.4 Softmax Regression — From Scratch

In [2]:
W = torch.normal(0, 0.01, size=(num_inputs, num_outputs), requires_grad=True)
b = torch.zeros(num_outputs, requires_grad=True)

def softmax(X):
    X_exp = torch.exp(X - X.max(dim=1, keepdim=True).values)  # subtract max: numerical stability
    partition = X_exp.sum(1, keepdim=True)
    return X_exp / partition  # broadcasting again, like Ch. 2

def net(X):
    return softmax(torch.matmul(X, W) + b)

def cross_entropy(y_hat, y):
    return -torch.log(y_hat[range(len(y_hat)), y])  # pick out the true class's predicted prob

def accuracy(y_hat, y):
    preds = y_hat.argmax(axis=1)
    return float((preds == y).sum()) / len(y)

> **`argmax(axis=1)` vs `max(dim=1)`** — `argmax` returns the *index* of the largest value per row (the predicted class); `max` returns the *value itself* (used above just for numerical stability, not for prediction).

In [3]:
def sgd(params, lr, batch_size):
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()

lr = 0.1
for epoch in range(5):
    total_loss, total_acc, n = 0.0, 0.0, 0
    for X, y in train_iter:
        y_hat = net(X)
        l = cross_entropy(y_hat, y)
        l.sum().backward()
        sgd([W, b], lr, len(y))
        total_loss += float(l.sum())
        total_acc += accuracy(y_hat, y) * len(y)
        n += len(y)
    print(f'epoch {epoch + 1}, loss {total_loss / n:.4f}, acc {total_acc / n:.4f}')

epoch 1, loss 0.0774, acc 0.9680
epoch 2, loss 0.0058, acc 1.0000
epoch 3, loss 0.0035, acc 1.0000
epoch 4, loss 0.0025, acc 1.0000
epoch 5, loss 0.0020, acc 1.0000


/tmp/ipykernel_500/2997678307.py:15: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:822.)
  total_loss += float(l.sum())


## 4.5 Concise Implementation
> **Key difference:** `nn.CrossEntropyLoss` combines softmax + log + NLL loss into one numerically-stable op. You feed it raw logits directly — no separate `softmax()` call, and no manual max-subtraction trick.

In [4]:
net2 = nn.Sequential(nn.Linear(num_inputs, num_outputs))
net2[0].weight.data.normal_(0, 0.01)
net2[0].bias.data.fill_(0)

loss_fn = nn.CrossEntropyLoss()
trainer = torch.optim.SGD(net2.parameters(), lr=0.1)

for epoch in range(5):
    total_loss, total_acc, n = 0.0, 0.0, 0
    for X, y in train_iter:
        logits = net2(X)
        l = loss_fn(logits, y)
        trainer.zero_grad()
        l.backward()
        trainer.step()
        total_loss += float(l) * len(y)
        total_acc += accuracy(logits, y) * len(y)
        n += len(y)
    print(f'epoch {epoch + 1}, loss {total_loss / n:.4f}, acc {total_acc / n:.4f}')

epoch 1, loss 0.0765, acc 0.9710
epoch 2, loss 0.0058, acc 1.0000
epoch 3, loss 0.0035, acc 1.0000
epoch 4, loss 0.0026, acc 1.0000
epoch 5, loss 0.0020, acc 1.0000


---
**KAN link:** the output layer here is still one linear map (`Xw+b`) feeding softmax. A KAN classifier would replace that final linear layer's edges with learned splines too — everything downstream (softmax, cross-entropy, `.backward()`) is unchanged, since none of it cares what kind of function produced the logits.